# **Phần 1: Phép khử Gauss và Các Ứng Dụng (Demo)**

**Thông tin nhóm:**
* **Môn học:** Toán Ứng Dụng và Thống Kê
* **Trường:** ĐH Khoa học Tự nhiên - ĐHQG TP.HCM
* **Nhóm:** 8
* **Lớp:** CQ2024/2
* **Thành viên:**
  1.   Nguyễn Đình Tuấn - 24120237
  2.   Nguyễn Anh Thái - 24120224
  3.   Nguyễn Huỳnh Gia Bảo - 24120264
  4.   Vòng Sau Hậu - 24120307
  5.   Lương Nhật Tân - 24120134

---

## **Giới thiệu tổng quan**
Notebook này demo các thuật toán nền tảng được cài đặt "từ đầu" (from scratch) bằng Python, bao gồm:
1. Giải hệ phương trình tuyến tính (Gauss Elimination & Back Substitution).
2. Tính định thức (Determinant).
3. Tìm ma trận nghịch đảo (Inverse via Gauss-Jordan).
4. Tìm hạng và các cơ sở (Rank, Column/Row/Null Spaces).
5. Kiểm chứng kết quả với thư viện NumPy.
...

In [ ]:
import numpy as np
import warnings

from gaussian import gaussian_eliminate, back_substitution
from determinant import copy_A, determinant
from inverse import inverse
from rank_and_basis import to_ref, to_rref, rank_and_basis

def print_matrix(name, M):
    print(f"{name}:")
    for row in M:
        print("  [" + "  ".join(f"{val:8.4f}" for val in row) + "]")

# 1. **Phép khử Gauss và Giải hệ phương trình**
Thuật toán sử dụng kỹ thuật **Partial Pivoting** để đảm bảo tính ổn định số học, tránh sai số khuếch đại khi gặp phần tử chốt quá nhỏ.

## **1.1. Test Case 1: Hệ phương trình cơ bản (Nghiệm duy nhất)**
Đây là trường hợp để kiểm tra xem thuật toán khử Gauss và thế ngược có chạy đúng logic không

In [ ]:
A=[[1, 2, -1], [2, 1, 1], [-1, 1, 0]]
b=[1, 8, 5]

M,x,s=gaussian_eliminate(A,b)
print(f'Ma tran sau khi khu: {M}')
print(f"Nghiem x: {x}")
print(f"So lan doi dong: {s}")

Ma tran sau khi khu: [[2, 1, 1, 8], [0.0, 1.5, -1.5, -3.0], [0.0, 0.0, 2.0, 12.0]]
Nghiem x: [-1.0, 4.0, 6.0]
So lan doi dong: 1


## **1.2. Test Case 2: Hệ vô nghiệm**
Kiểm tra logic phát hiện dòng mâu thuẫn ($0x = c$ với $c \neq 0$)

In [ ]:
A=[[1, 1, 1], [2, 2, 2], [1, 2, 3]]
b=[1, 5, 3]
M,x,s=gaussian_eliminate(A,b)
print(f'Ma tran sau khi khu: {M}')
print(f"Nghiem x: {x}")
print(f"So lan doi dong: {s}")

Ma tran sau khi khu: [[2, 2, 2, 5], [0.0, 1.0, 2.0, 0.5], [0.0, 0.0, 0.0, -1.5]]
Nghiem x: Vo nghiem
So lan doi dong: 2


## **1.3. Test case 3: Vô số nghiệm (Có ẩn cố định và ẩn tự do)**

In [ ]:
A=[[1, 1, 1], [0, 0, 1]]
b=[2, 1]
M,x,s=gaussian_eliminate(A,b)
print(f'Ma tran sau khi khu: {M}')
print(f"Nghiem x: {x}")
print(f"So lan doi dong: {s}")

Ma tran sau khi khu: [[1, 1, 1, 2], [0.0, 0.0, 1.0, 1.0]]
Nghiem x: ['(1.0/1 - 1*x2)/1', 'x2', 1.0]
So lan doi dong: 0


## **1.4. Test case 4: Ma trận chữ nhật (n < m)**
Kiểm tra tính ổn định và chính xác khi làm việc với ma trận hình chữ nhật.

In [ ]:
A=[[1, 2, 3, 4], [2, 4, 8, 10]]
b=[1, 2]
M,x,s=gaussian_eliminate(A,b)
print(f'Ma tran sau khi khu: {M}')
print(f"Nghiem x: {x}")
print(f"So lan doi dong: {s}")

Ma tran sau khi khu: [[2, 4, 8, 10, 2], [0.0, 0.0, -1.0, -1.0, 0.0]]
Nghiem x: ['(2/2 - 4*x2 - 10*x4)/2', 'x2', '(0.0/-1.0 + 1.0*x4)/-1.0', 'x4']
So lan doi dong: 1


## **1.5. Test case 5: Ma trận cần Partial Pivoting (Phần tử chốt bằng 0)**
Kiểm tra tính ổn định số học khi hàng đầu tiên có pivot bằng 0

In [ ]:
A=[[0, 2, 1], [1, -2, -3], [5, -1, 2]]
b=[4, 0, -3]
M,x,s=gaussian_eliminate(A,b)
print(f'Ma tran sau khi khu: {M}')
print(f"Nghiem x: {x}")
print(f"So lan doi dong: {s}")

Ma tran sau khi khu: [[5, -1, 2, -3], [0.0, 2.0, 1.0, 4.0], [0.0, 0.0, -2.5, 4.2]]
Nghiem x: [0.64, 2.84, -1.6800000000000002]
So lan doi dong: 2


## **1.6. Test case 6: Ma trận có pivot nhỏ**

In [ ]:
A = [[0.0000000000001, 1], [0.0000000000002, 1]]
b = [1, 2]
M,x,s=gaussian_eliminate(A,b)
print(f'Ma tran sau khi khu: {M}')
print(f"Nghiem x: {x}")
print(f"So lan doi dong: {s}")

Ma tran sau khi khu: [[2e-13, 1, 2], [0.0, 0.5, 0.0]]
Nghiem x: [10000000000000.0, 0.0]
So lan doi dong: 1


/content/drive/MyDrive/Colab Notebooks/TUDTK_Do an 1/gaussian.py:26: UserWarning: Pivot tai cot 0 gan bang 0 (2.00e-13). He co the ill-conditioned.
  warnings.warn(f"Pivot tai cot {pivot_col} gan bang 0 ({max_val:.2e}). He co the ill-conditioned.")


# 2. **Tính Định thức ma trận**
Định thức được tính bằng tích các phần tử đường chéo sau khi đưa về dạng tam giác trên (REF), có xét đến dấu $(-1)^s$ dựa trên số lần đổi dòng.

## **2.1. Test Case 1: Ma trận đơn vị (Identity Matrix)**
Kết quả kỳ vọng: 1.0

In [ ]:
tc1 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
actual = determinant(tc1)
expected = 1.0
print(f"Kết quả thực tế : {actual}")
print(f"Kết quả kỳ vọng : {expected}")
if abs(actual - expected) < 1e-8:
    print("✓ PASS")
else:
    print("✗ FAIL")

Kết quả thực tế : 1.0
Kết quả kỳ vọng : 1.0
✓ PASS


## **2.2. Test Case 2: Ma trận suy biến (Singular Matrix)**
Kết quả kỳ vọng: 0

In [ ]:
tc2 = [[1, 2, 3], [4, 5, 6], [5, 7, 9]]
actual = determinant(tc2)
expected = 0.0
print(f"Kết quả thực tế : {actual}")
print(f"Kết quả kỳ vọng : {expected}")
if abs(actual - expected) < 1e-8:
    print("✓ PASS")
else:
    print("✗ FAIL")

Kết quả thực tế : 0
Kết quả kỳ vọng : 0.0
✓ PASS


## **2.3. Test Case 3: Ma trận cần hoán vị dòng (Cần Partial Pivoting)**
Kết quả kỳ vọng: -2.0

In [ ]:
tc3 = [[0, 1], [2, 0]]
actual = determinant(tc3)
expected = -2.0
print(f"Kết quả thực tế : {actual}")
print(f"Kết quả kỳ vọng : {expected}")
if abs(actual - expected) < 1e-8:
    print("✓ PASS")
else:
    print("✗ FAIL")

Kết quả thực tế : -2.0
Kết quả kỳ vọng : -2.0
✓ PASS


## **2.4. Test Case 4: Ma trận với số thực rất nhỏ**
Kết quả kỳ vọng: 1e-10

In [ ]:
tc4 = [[1e-5, 0], [0, 1e-5]]
actual = determinant(tc4)
expected = 1e-10
print(f"Kết quả thực tế : {actual}")
print(f"Kết quả kỳ vọng : {expected}")
if abs(actual - expected) < 1e-8:
    print("✓ PASS")
else:
    print("✗ FAIL")

Kết quả thực tế : 1.0000000000000002e-10
Kết quả kỳ vọng : 1e-10
✓ PASS


## **2.5. Test Case 5: Ma trận Hilbert 3×3 (ill-conditioned)**
Kết quả kỳ vọng: ~0.00046296296

In [ ]:
tc5 = [[1.0, 0.5, 1.0/3.0], [0.5, 1.0/3.0, 0.25], [1.0/3.0, 0.25, 0.2]]
actual = determinant(tc5)
expected = 0.00046296296
print(f"Kết quả thực tế : {actual}")
print(f"Kết quả kỳ vọng : {expected}")
if abs(actual - expected) < 1e-8:
    print("✓ PASS")
else:
    print("✗ FAIL")

Kết quả thực tế : 0.00046296296296296135
Kết quả kỳ vọng : 0.00046296296
✓ PASS


# 3. **Tìm ma trận nghịch đảo bằng Gauss-Jordan**
Sử dụng phương pháp biến đổi dòng đồng thời trên ma trận ghép $[A|I]$.

## **3.1. Test case 1: Ma trận đơn vị**

In [ ]:
A = [[1, 0, 0],
     [0, 1, 0],
     [0, 0, 1]]
res =inverse(A)
if res:
    for row in res:
        print(" ".join(f"{x:>8.2f}" for x in row))
else:
    print("ma trận không khả nghịch")

    1.00     0.00     0.00
    0.00     1.00     0.00
    0.00     0.00     1.00


## **3.2. Test case 2: Ma trận suy biến**

In [ ]:
A = [[1, 2, 3],
     [4, 5, 6],
     [7, 8, 9]]
res =inverse(A)
if res:
    for row in res:
        print(" ".join(f"{x:>8.2f}" for x in row))
else:
    print("ma trận không khả nghịch")

ma trận không khả nghịch


## **3.3. Test case 3: Ma trận đường chéo rỗng**

In [ ]:
A = [[1e-8, 1, 2],
     [1, 1e-8, 3],
     [2, 3, 1e-8]]
res =inverse(A)
if res:
    for row in res:
        print(" ".join(f"{x:>8.2f}" for x in row))
else:
    print("ma trận không khả nghịch")

   -0.75     0.50     0.25
    0.50    -0.33     0.17
    0.25     0.17    -0.08


## **3.4. Test case 4: Ma trận Hilbert**

In [ ]:
A = [ [1.0, 1/2, 1/3],
      [1/2, 1/3, 1/4],
      [1/3, 1/4, 1/5]]

res =inverse(A)
if res:
    for row in res:
        print(" ".join(f"{x:>8.2f}" for x in row))
else:
    print("ma trận không khả nghịch")

    9.00   -36.00    30.00
  -36.00   192.00  -180.00
   30.00  -180.00   180.00


## **3.5. Test case 5: Ma trận hoán vị xung quanh**

In [ ]:
A = [[1, 2, 3],
     [3, 1, 2],
     [2, 3, 1]]
print ("test 5: ")
res =inverse(A)
if res:
    for row in res:
        print(" ".join(f"{x:>8.2f}" for x in row))
else:
    print("ma trận không khả nghịch")

test 5: 
   -0.28     0.39     0.06
    0.06    -0.28     0.39
    0.39     0.06    -0.28


# 4. **Tìm hạng và các cơ sở**
Dựa trên dạng bậc thang rút gọn (RREF) để xác định biến tự do và các vector cơ sở của không gian Dòng, Cột và Không gian nghiệm (Null space).

## **4.1. Ma trận vuông nxn, rank = n**

In [ ]:
A=[
    [1,2,3],
    [0,1,4],
    [5,6,0]
]
rank, col_basis,row_basis, null_basis = rank_and_basis(A)
print(f"Hang cua ma tran: {rank}")
print(f"Co so dong: {row_basis}")
print(f"Co so cot: {col_basis}")
print(f"Co so nghiem: {null_basis}")

Hang cua ma tran: 3
Co so dong: [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
Co so cot: [[1, 0, 5], [2, 1, 6], [3, 4, 0]]
Co so nghiem: []


## **4.2. Ma trận nxn, rank < n**

In [ ]:
A=[
    [1,2,3],
    [4,5,6],
    [7,8,9]
]
rank, col_basis, row_basis, null_basis = rank_and_basis(A)
print(f"Hang cua ma tran: {rank}")
print(f"Co so dong: {row_basis}")
print(f"Co so cot: {col_basis}")
print(f"Co so nghiem: {null_basis}")

Hang cua ma tran: 2
Co so dong: [[1.0, 0.0, -1.0], [0.0, 1.0, 2.0]]
Co so cot: [[1, 4, 7], [2, 5, 8]]
Co so nghiem: [[1.0, -2.0, 1]]


## **4.3. Ma trận mxn, m < n**

In [ ]:
A=[
    [1,-2,0,4],
    [3, 1,1,0]
]
rank, col_basis, row_basis, null_basis = rank_and_basis(A)
print(f"Hang cua ma tran: {rank}")
print(f"Co so dong: {row_basis}")
print(f"Co so cot: {col_basis}")
print(f"Co so nghiem: {null_basis}")

Hang cua ma tran: 2
Co so dong: [[1.0, 0.0, 0.28571428571428575, 0.5714285714285714], [0.0, 1.0, 0.14285714285714285, -1.7142857142857142]]
Co so cot: [[1, 3], [-2, 1]]
Co so nghiem: [[-0.28571428571428575, -0.14285714285714285, 1, 0], [-0.5714285714285714, 1.7142857142857142, 0, 1]]


## **4.4. Ma trận mxn, m > n**

In [ ]:
A=[
    [1,2],
    [3,4],
    [5,6]
]
rank, col_basis, row_basis, null_basis = rank_and_basis(A)
print(f"Hang cua ma tran: {rank}")
print(f"Co so dong: {row_basis}")
print(f"Co so cot: {col_basis}")
print(f"Co so nghiem: {null_basis}")

Hang cua ma tran: 2
Co so dong: [[1.0, 0.0], [0.0, 1.0]]
Co so cot: [[1, 3, 5], [2, 4, 6]]
Co so nghiem: []
